# MarkItDown in miniature: interactive walkthrough

`demo.py` runs seven scenes end to end. This notebook takes the same machine apart and runs each stage on its own: the converter contract, the hints, the guessing, the registry, the dispatch loop, recursion, and plugins.

Some cells reach into private names (`md._converters`, `md._guesses`, `core._sniff`) on purpose. They are the stages that the public `convert()` chains together:

```
source ─► (stream, StreamInfo) ─► guesses ─► registry by priority ─► accepts? ─► convert? ─► first success wins
```

In [1]:
import io
import logging
import sys
import zipfile
from contextlib import contextmanager
from importlib.metadata import entry_points

import minimd._markitdown as core  # the orchestrator module, for its private stages
import minimd.converters as converters
from demo import (
    SAMPLES,
    IndentByDepth,
    make_zip,
)  # reuse the demo's fixtures and trace indent
from minimd import (
    DocumentConverter,
    DocumentConverterResult,
    FileConversionException,
    MarkItDown,
    StreamInfo,
)


@contextmanager
def traced():
    """Print minimd's dispatch trace inside the block, indented by recursion depth."""
    handler = logging.StreamHandler(sys.stdout)
    handler.setFormatter(logging.Formatter("%(message)s"))
    handler.addFilter(IndentByDepth())
    logger = logging.getLogger("minimd")
    logger.addHandler(handler)
    logger.setLevel(logging.DEBUG)
    try:
        yield
    finally:
        logger.removeHandler(handler)


def stream(data: bytes | str) -> io.BytesIO:
    return io.BytesIO(data.encode() if isinstance(data, str) else data)


def show_registry(md: MarkItDown) -> None:
    """The list as stored (newest first), next to the order dispatch tries it."""
    stored = [type(r.converter).__name__ for r in md._converters]
    tried = sorted(md._converters, key=lambda r: r.priority)
    print(f"{'stored (newest first)':<24}   dispatch order")
    for name, r in zip(stored, tried):
        print(f"{name:<24}   {r.priority:+5.1f}  {type(r.converter).__name__}")

## 1. The contract: two methods, one signature

A converter works on its own. `accepts()` judges from the hints alone, and `convert()` does the work. The dispatcher only adds ordering and an error policy around them.

In [2]:
converter = converters.CsvConverter()
data = stream("modality,studies\nCT,412\n")

print("accepts, hinted .csv:", converter.accepts(data, StreamInfo(extension=".csv")))
print("accepts, no hints:   ", converter.accepts(data, StreamInfo()))
print(converter.convert(data, StreamInfo(extension=".csv")))

accepts, hinted .csv: True
accepts, no hints:    False
| modality | studies |
| --- | --- |
| CT | 412 |


With no hints, `accepts()` said no, although the bytes are a valid CSV. Converters don't inspect bytes to find their type. That job belongs to the dispatcher (section 3), which does it once for everyone.

## 2. `StreamInfo`: hints as an immutable value

`copy_and_update()` builds a new guess. A non-None field from another `StreamInfo` wins, a `None` never erases anything, and keyword arguments override outright.

In [3]:
base = StreamInfo(extension=".txt", filename="notes.txt")

print(base.copy_and_update(StreamInfo(mimetype="text/plain", extension=None)))
print(base.copy_and_update(extension=".zip"))
try:
    base.extension = ".csv"
except Exception as error:
    print(f"{type(error).__name__}: {error}")

StreamInfo(mimetype='text/plain', extension='.txt', charset=None, filename='notes.txt', local_path=None, url=None)
StreamInfo(mimetype=None, extension='.zip', charset=None, filename='notes.txt', local_path=None, url=None)
FrozenInstanceError: cannot assign to field 'extension'


Frozen matters because, in the next step, one guess fans out into several. No step can change a guess that another step still holds.

## 3. Guessing: what the name says vs what the bytes say

`_sniff()` stands in for magika. It looks only at the bytes and gives one of three answers: a magic number it knows, "this is UTF-8 text", or no opinion (`None`).

In [4]:
zip_bytes = make_zip()
samples = [
    ("zip archive", zip_bytes),
    ("utf-8 text", b"hello"),
    ("unknown binary", b"\x00\xff\xfe"),
]
for label, data in samples:
    print(f"{label:<15} → {core._sniff(stream(data))}")

zip archive     → StreamInfo(mimetype='application/zip', extension='.zip', charset=None, filename=None, local_path=None, url=None)
utf-8 text      → StreamInfo(mimetype=None, extension=None, charset='utf-8', filename=None, local_path=None, url=None)
unknown binary  → None


`_guesses()` cross-checks the sniffer's answer against the hints. If they agree, you get one merged guess. If they conflict, you get both, with the hints' guess first.

In [5]:
md = MarkItDown()
cases = {
    "report.csv on disk": (
        (SAMPLES / "report.csv").read_bytes(),
        StreamInfo(extension=".csv", filename="report.csv"),
    ),
    "zip, no hints": (zip_bytes, StreamInfo()),
    "zip named notes.txt": (
        zip_bytes,
        StreamInfo(extension=".txt", filename="notes.txt"),
    ),
}
for label, (data, hints) in cases.items():
    print(label)
    for n, g in enumerate(md._guesses(stream(data), hints), start=1):
        print(
            f"  guess {n}: extension={g.extension} mimetype={g.mimetype} charset={g.charset}"
        )

report.csv on disk
  guess 1: extension=.csv mimetype=text/csv charset=utf-8
zip, no hints
  guess 1: extension=.zip mimetype=application/zip charset=None
zip named notes.txt
  guess 1: extension=.txt mimetype=text/plain charset=None
  guess 2: extension=.zip mimetype=application/zip charset=None


The third case is demo scene 4: two guesses, so dispatch gets a second chance when the name lies. Dispatch also appends one empty `StreamInfo()` as a last resort.

## 4. The registry: order is data

`register_converter()` puts each registration at the front of a list (`insert(0)`), and dispatch sorts that list by priority with a stable sort. The result: a lower priority goes first, and among equal priorities the newest registration goes first.

Add a generic converter of your own at the catch-all priority, 10:

In [6]:
class LogConverter(DocumentConverter):
    """A user's converter: wraps .log files in a code fence."""

    def accepts(self, file_stream, stream_info, **kwargs):
        return (stream_info.extension or "") == ".log"

    def convert(self, file_stream, stream_info, **kwargs):
        fence = "```"
        return DocumentConverterResult(
            f"{fence}\n{file_stream.read().decode()}\n{fence}"
        )


md.register_converter(LogConverter(), priority=core.PRIORITY_GENERIC_FILE_FORMAT)
show_registry(md)

stored (newest first)      dispatch order
LogConverter                +0.0  YamlConverter
YamlConverter               +0.0  CsvConverter
CsvConverter               +10.0  LogConverter
ZipConverter               +10.0  ZipConverter
PlainTextConverter         +10.0  PlainTextConverter


`LogConverter` is stored first, and the sort moves it behind the 0.0 converters but ahead of the other 10.0 ones. PlainText would accept a `.log` too, since the sniffer gives it a charset, but it never gets the chance:

In [7]:
with traced():
    log_text = "12:01 study received\n12:03 report signed"
    print(md.convert(stream(log_text), stream_info=StreamInfo(extension=".log")))

    guess 1: extension=.log mimetype=text/plain charset=utf-8


       +0.0  YamlConverter         no


       +0.0  CsvConverter          no


      +10.0  LogConverter          yes → ✓ converted


```
12:01 study received
12:03 report signed
```


## 5. Dispatch: a failure is a record, not an exit

Demo scene 4 again, with the trace on. Guess 1 trusts the name: plain text accepts the input and fails to decode it. The failure is recorded, the stream rewound, and guess 2 (the bytes) succeeds. The indented block is the zip's members going through dispatch themselves.

In [8]:
with traced():
    result = md.convert(
        stream(zip_bytes),
        stream_info=StreamInfo(extension=".txt", filename="notes.txt"),
    )
print(result)

    guess 1: extension=.txt mimetype=text/plain charset=None


       +0.0  YamlConverter         no


       +0.0  CsvConverter          no


      +10.0  LogConverter          no


      +10.0  ZipConverter          no


...

          +10.0  LogConverter          no


          +10.0  ZipConverter          no


          +10.0  PlainTextConverter    no


      +10.0  ZipConverter          yes → ✓ converted


## File: report.csv

| modality | studies | median_tat_min |
| --- | --- | --- |
| CT | 412 | 38 |
| MRI | 157 | 95 |
| X-ray | 1290 | 12 |

## File: readme.txt

Monthly study volumes. See report.csv.


When nobody succeeds, the records decide the error. `FileConversionException` keeps them as `.attempts`, so the caller can see who tried and why each one failed:

In [9]:
try:
    md.convert(
        stream(b"PK\x03\x04 cut short"), stream_info=StreamInfo(extension=".zip")
    )
except FileConversionException as error:
    for attempt in error.attempts:
        print(f"{type(attempt.converter).__name__} → {attempt.error!r}")

ZipConverter → BadZipFile('File is not a zip file')


When a failure is followed by a success, the failure is dropped without a word. That is demo scene 5, with `config.yaml`. One catch: Jupyter installs `pyyaml` for its own use (`jupyter_events` needs it), so this kernel can probably import `yaml`:

In [10]:
print("yaml importable in this kernel:", converters._yaml_import_error is None)

yaml importable in this kernel: True


`YamlConverter` decides only from the import error it stored when the module loaded. Swap in a fake one to see the soft failure, then put the real state back:

In [11]:
real_error = converters._yaml_import_error
converters._yaml_import_error = ImportError("No module named 'yaml'")
try:
    with traced():
        print(md.convert(SAMPLES / "config.yaml"))
finally:
    converters._yaml_import_error = real_error

    guess 1: extension=.yaml mimetype=None charset=utf-8


       +0.0  YamlConverter         yes → ✗ MissingDependencyException


       +0.0  CsvConverter          no


      +10.0  LogConverter          no


      +10.0  ZipConverter          no


      +10.0  PlainTextConverter    yes → ✓ converted


model: ct-brain-hemorrhage
version: 2.1
threshold: 0.7
labels: [ich, sah, sdh]


In [12]:
with traced():
    print(md.convert(SAMPLES / "config.yaml"))

    guess 1: extension=.yaml mimetype=None charset=utf-8


       +0.0  YamlConverter         yes → ✓ converted


- **model**: ct-brain-hemorrhage
- **version**: 2.1
- **threshold**: 0.7
- **labels**: ['ich', 'sah', 'sdh']


## 6. Recursion: a zip inside a zip

`ZipConverter` sends each member back through `convert_stream()`, the same front door any caller uses. Nesting needs no extra code: each level is one more dispatch, which shows as one more indent in the trace.

The headings come out flat: `inner.zip` and its own `report.csv` are both `##`. markitdown's `ZipConverter` flattens them the same way.

In [13]:
inner = io.BytesIO()
with zipfile.ZipFile(inner, "w") as archive:
    archive.write(SAMPLES / "report.csv", "report.csv")

outer = io.BytesIO()
with zipfile.ZipFile(outer, "w") as archive:
    archive.writestr("inner.zip", inner.getvalue())
    archive.writestr("notes.md", "Batch 7: one archive inside another.")

with traced():
    print(md.convert(stream(outer.getvalue())))

    guess 1: extension=.zip mimetype=application/zip charset=None


       +0.0  YamlConverter         no


       +0.0  CsvConverter          no


      +10.0  LogConverter          no


        guess 1: extension=.zip mimetype=application/zip charset=None


...

          +10.0  LogConverter          no


          +10.0  ZipConverter          no


          +10.0  PlainTextConverter    yes → ✓ converted


      +10.0  ZipConverter          yes → ✓ converted


## File: inner.zip

## File: report.csv

| modality | studies | median_tat_min |
| --- | --- | --- |
| CT | 412 | 38 |
| MRI | 157 | 95 |
| X-ray | 1290 | 12 |

## File: notes.md

Batch 7: one archive inside another.


## 7. Plugins: found by metadata, loaded lazily, opt-in

The core reads one entry-point group from the installed packages' metadata. That is all it knows about plugins. So far nothing in this kernel has asked for them:

In [14]:
print(entry_points(group="minimd.plugin"))
print("plugins loaded yet:        ", core._plugins)
print("plugin module imported yet:", "minimd_sample_plugin" in sys.modules)

EntryPoints((EntryPoint(name='sample_plugin', value='minimd_sample_plugin', group='minimd.plugin'),))
plugins loaded yet:         None
plugin module imported yet: False


`enable_plugins=True` loads the plugin once per process and calls its `register_converters()`, which goes through the same public `register_converter()` that section 4 used:

In [15]:
plugged = MarkItDown(enable_plugins=True)
print("plugins loaded:        ", [plugin.__name__ for plugin in core._plugins])
print("plugin module imported:", "minimd_sample_plugin" in sys.modules, end="\n\n")
show_registry(plugged)

plugins loaded:         ['minimd_sample_plugin']
plugin module imported: True

stored (newest first)      dispatch order
SniffingCsvConverter        -1.0  SniffingCsvConverter
IniConverter                +0.0  IniConverter
YamlConverter               +0.0  YamlConverter
CsvConverter                +0.0  CsvConverter
ZipConverter               +10.0  ZipConverter
PlainTextConverter         +10.0  PlainTextConverter


The shadow at -1.0 now sees every CSV first. The built-in `CsvConverter` is still registered, and answers only if the shadow raises. Here is `supplies.csv` (`;` delimiters, decimal commas) without the plugin, then with it:

In [16]:
print(md.convert(SAMPLES / "supplies.csv"), end="\n\n")
with traced():
    print(plugged.convert(SAMPLES / "supplies.csv"))

| item;unit_price_eur |
| --- |
| contrast agent;12 | 50 |
| syringe;0 | 80 |
| lead apron;149 | 00 |

    guess 1: extension=.csv mimetype=text/csv charset=utf-8


       -1.0  SniffingCsvConverter  yes → ✓ converted


| item | unit_price_eur |
| --- | --- |
| contrast agent | 12,50 |
| syringe | 0,80 |
| lead apron | 149,00 |


## Where next

- `README.md`: the ten ideas, each linked to the matching lines in markitdown, and five exercises.
- `demo.py`: the seven scenes in one run (`uv run --exact demo.py`).